## Bevezetés 

Ez a lecke a következőket fogja bemutatni: 
- Mi az a függvényhívás és milyen felhasználási esetei vannak 
- Hogyan lehet Azure OpenAI segítségével függvényhívást létrehozni 
- Hogyan integráljunk egy függvényhívást egy alkalmazásba 

## Tanulási célok 

A lecke elvégzése után tudni fogod és érted majd: 

- A függvényhívás használatának célját 
- A függvényhívás beállítása az Azure Open AI szolgáltatással 
- Hatékony függvényhívások tervezése az alkalmazásod felhasználási esetéhez 


## A függvényhívások megértése 

Ehhez a leckéhez egy olyan funkciót szeretnénk fejleszteni az oktatási startupunk számára, amely lehetővé teszi a felhasználók számára, hogy chatbot segítségével találjanak műszaki kurzusokat. Ajánlani fogunk olyan kurzusokat, amelyek megfelelnek a képességszintjüknek, jelenlegi szerepüknek és az érdeklődési körükben lévő technológiáknak. 

Ehhez a következő kombinációt fogjuk használni: 
 - Az `Azure Open AI` használata a felhasználói beszélgetési élmény létrehozásához
 - A `Microsoft Learn Catalog API` használata, hogy a felhasználók megtalálják a számukra megfelelő kurzusokat a kérésük alapján 
 - A `Function Calling` használata, hogy a felhasználó lekérdezését egy függvényhez küldjük az API kérés kivitelezéséhez. 

Kezdésként nézzük meg, miért szeretnénk egyáltalán használni a függvényhívást: 


### Miért a függvényhívás 

Ha elvégeztél bármely más leckét ebben a tanfolyamban, valószínűleg érted a Nagynyelvű Modellek (LLM-ek) használatának erejét. Remélhetőleg a korlátaikat is látod. 

A függvényhívás egy olyan Azure Open AI Szolgáltatás funkciója, amely az alábbi korlátokat hivatott áthidalni: 
1) Következetes válaszformátum 
2) Az alkalmazás egyéb adatforrásainak használata csevegési kontextusban 

A függvényhívás előtt az LLM válaszai struktúrálatlanok és következetlenek voltak. A fejlesztőknek bonyolult ellenőrző kódokat kellett írniuk, hogy minden válaszvariációt kezelni tudjanak. 

A felhasználók nem kaphattak válaszokat olyan kérdésekre, hogy „Milyen az aktuális időjárás Stockholmban?”. Ez azért volt, mert a modellek csak a tanításuk idején meglévő adatokra korlátozódtak. 

Nézzük meg az alábbi példát, amely ezt a problémát szemlélteti: 

Tegyük fel, hogy létre akarunk hozni egy diákadatbázist, hogy megfelelő kurzusokat javasolhassunk számukra. Lent két leírást látunk diákokról, amelyek nagyon hasonló adatokat tartalmaznak.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Ezt egy LLM-nek szeretnénk elküldeni, hogy elemezze az adatokat. Ezt később felhasználhatjuk az alkalmazásunkban, hogy elküldjük egy API-nak vagy elmentsük egy adatbázisban. 

Készítsünk két azonos promptot, amelyben utasítjuk az LLM-et, hogy milyen információkra vagyunk kíváncsiak: 


Ezt szeretnénk elküldeni egy LLM-nek, hogy elemezze azokat a részeket, amik fontosak a termékünk szempontjából. Így két azonos promptot hozhatunk létre az LLM számára az utasításhoz: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Miután elkészítettük ezt a két promptot, elküldjük őket az LLM-nek a `client.responses.create` használatával. A promptot az `input` változóban tároljuk, és a szerepet `user`-re állítjuk. Ez azt hivatott utánozni, mintha egy felhasználó üzenetet írna egy chatbotnak.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Most már mindkét kérést elküldhetjük az LLM-nek, és megvizsgálhatjuk a választ, amit kapunk. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Bár a felhívások azonosak, és a leírások hasonlóak, különböző formátumokat kaphatunk a `Grades` tulajdonságból.

Ha a fenti cellát többször lefuttatjuk, a formátum lehet `3.7` vagy `3.7 GPA`.

Ennek oka, hogy a LLM nem strukturált adatokat vesz be a megírt felhívás formájában, és szintén nem strukturált adatokat ad vissza. Strukturált formátumban kell lennie az adatoknak, hogy tudjuk, mire számítsunk az adatok tárolásakor vagy használatakor.

Funkcionális hívás használatával biztosíthatjuk, hogy strukturált adatokat kapjunk vissza. Funkcióhívás esetén a LLM valójában nem hív meg vagy futtat semmilyen funkciót. Ehelyett egy struktúrát hozunk létre, amelyet a LLM követhet a válaszaiban. Ezután ezeket a strukturált válaszokat használjuk arra, hogy tudjuk, melyik funkciót kell futtatni az alkalmazásainkban.
 


![Függvényhívási folyamatábra](../../../../translated_images/hu/Function-Flow.083875364af4f4bb.webp)


Ezután azt, amit a függvény visszaad, elküldhetjük vissza az LLM-nek. Az LLM ezután természetes nyelven válaszol a felhasználó kérdésére. 


### Funkcióhívások használati esetei

**Külső eszközök hívása**
A chatbotok remekül válaszolnak a felhasználók kérdéseire. A funkcióhívás használatával a chatbotok a felhasználói üzenetekből kiindulva bizonyos feladatokat elvégezhetnek. Például egy diák megkérheti a chatbotot, hogy „Küldj e-mailt az oktatómnak, hogy több segítségre van szükségem ebben a tárgyban”. Ez egy funkcióhívást tehet a `send_email(to: string, body: string)` függvényre.


**API vagy adatbázis-lekérdezések létrehozása**
A felhasználók természetes nyelven találnak információt, amely egy formázott lekérdezéssé vagy API-kéréssé alakul. Ennek egy példája lehet egy tanár, aki megkérdezi: „Kik azok a diákok, akik befejezték az utolsó házi feladatot”, ami meghívhat egy `get_completed(student_name: string, assignment: int, current_status: string)` nevű függvényt.


**Strukturált adatok létrehozása**
A felhasználók egy szövegrészt vagy CSV-t vehetnek, és a LLM segítségével kinyerhetik belőle a fontos információkat. Például egy diák egy Wikipedia-cikket a békeszerződésekről alakíthat át AI tanulókártyák készítéséhez. Ezt egy `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` nevű funkció használatával lehet megtenni.


## 2. Az első függvényhívás létrehozása 

A függvényhívás létrehozásának folyamata 3 fő lépést tartalmaz: 
1. A Chat Completions API hívása a függvényeid listájával és egy felhasználói üzenettel 
2. A modell válaszának elolvasása, hogy végrehajts egy műveletet, például egy függvény vagy API hívás végrehajtását 
3. Egy másik hívás végrehajtása a Chat Completions API-hoz a függvényed válaszával, hogy ezt az információt felhasználva választ készíts a felhasználónak. 


![Egy függvényhívás folyamata](../../../../translated_images/hu/LLM-Flow.3285ed8caf4796d7.webp)


### Egy függvényhívás elemei 

#### Felhasználói bemenet 

Az első lépés egy felhasználói üzenet létrehozása. Ezt dinamikusan is hozzárendelhetjük egy szöveges bemenet értékének felvételével, vagy itt is rendelhetünk értéket. Ha ez az első alkalom, hogy a Chat Completions API-val dolgozik, meg kell határoznunk az üzenet `role` és `content` mezőjét. 

A `role` lehet `system` (szabályok létrehozása), `assistant` (a modell) vagy `user` (a végfelhasználó). A függvényhívásnál ezt `user`-nek állítjuk be, és példaként megadunk egy kérdést. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Függvények létrehozása. 

Ezután definiálunk egy függvényt és annak paramétereit. Itt csak egy `search_courses` nevű függvényt fogunk használni, de több függvényt is létrehozhatsz.

**Fontos** : A függvények a rendszerüzenetben szerepelnek az LLM-nek, és beleszámítanak az elérhető tokenek mennyiségébe. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Meghatározások** 

`name` - Az a függvénynév, amelyet hívni szeretnénk. 

`description` - Ez a függvény működésének leírása. Fontos, hogy itt pontos és világos legyen. 

`parameters` - Egy lista azokról az értékekről és formátumról, amelyeket a modellnek a válaszában elő kell állítania. 


`type` - Az adattípus, amelyben a tulajdonságok tárolva lesznek. 

`properties` - A specifikus értékek listája, amelyeket a modell a válaszában használni fog. 


`name` - a tulajdonság neve, amelyet a modell a formázott válaszában használ. 

`type` - Ennek a tulajdonságnak az adattípusa. 

`description` - A specifikus tulajdonság leírása. 


**Opcionális**

`required` - A függvényhívás befejezéséhez szükséges tulajdonság. 


### A függvényhívás elkészítése  
Miután definiáltuk a függvényt, most be kell illesztenünk a Chat Completion API hívásába. Ezt úgy tesszük, hogy hozzáadjuk a `functions` elemet a kéréshez. Ebben az esetben `functions=functions`.  

Van egy lehetőség arra is, hogy a `function_call`-t `auto` értékre állítsuk. Ez azt jelenti, hogy hagyjuk, hogy az LLM döntsön arról, melyik függvényt hívja meg a felhasználói üzenet alapján, ahelyett, hogy mi rendelnénk hozzá.  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Most nézzük meg a választ, és lássuk, hogyan van formázva: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Látható, hogy a függvény neve meg van hívva, és a felhasználói üzenet alapján a LLM képes volt megtalálni az adatokat, amelyek megfelelnek a függvény argumentumainak. 


## 3. Függvényhívások integrálása egy alkalmazásba.


Miután leteszteltük a formázott választ az LLM-ből, most integrálhatjuk ezt egy alkalmazásba.

### A folyamat kezelése

Ahhoz, hogy ezt integráljuk az alkalmazásunkba, tegyük meg a következő lépéseket:

Először is, hívjuk meg az Open AI szolgáltatásokat, és tároljuk az üzenetet egy `response_message` nevű változóban.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Most definiáljuk azt a függvényt, amely meghívja a Microsoft Learn API-t, hogy lekérjen egy tanfolyamlistát: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Jó gyakorlatként először megnézzük, hogy a modell szeretne-e egy függvényt meghívni. Ezután létrehozunk az elérhető függvények közül egyet, és összehangoljuk azt a hívott függvénnyel.
Ezután a függvény argumentumait hozzárendeljük az LLM-ből származó argumentumokhoz.

Végül hozzáadjuk a függvényhívás üzenetét és azokat az értékeket, amelyeket a `search_courses` üzenet adott vissza. Ez megadja az LLM-nek az összes szükséges információt ahhoz, hogy
természetes nyelven válaszoljon a felhasználónak.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Most elküldjük a frissített üzenetet az LLM-nek, hogy természetes nyelvű válaszban részesüljünk az API JSON formátumú válasza helyett. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kódkihívás 

Nagyszerű munka! Az Azure Open AI Function Calling további elsajátításához építheted ezt: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - A funkció több paramétere, amelyek segíthetnek a tanulóknak több kurzus megtalálásában. Az elérhető API paramétereket itt találod: 
 - Hozz létre egy másik függvényhívást, amely több információt vesz fel a tanulótól, például az anyanyelvét 
 - Készíts hibakezelést arra az esetre, ha a függvényhívás és/vagy az API hívás nem ad vissza megfelelő kurzusokat 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
